---
## §0. Configuration

In [1]:
import json, os
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR   = Path("/kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data")
OUTPUT_DIR = Path("/kaggle/working")

TRAIN_PATH     = DATA_DIR / "train.csv"
VAL_INPUT_PATH = DATA_DIR / "validation_input.csv"
VAL_INDEX_PATH = DATA_DIR / "forecast_index_validation.csv"

EXP_ID         = "13a_tft_multiseed"
OOF_OUT        = OUTPUT_DIR / f"oof_predictions_{EXP_ID}.csv"
VAL_OUT        = OUTPUT_DIR / f"validation_predictions_{EXP_ID}.csv"
PROBE_OUT      = OUTPUT_DIR / f"probe_{EXP_ID}.csv"
IMPORTANCE_OUT = OUTPUT_DIR / f"variable_importance_{EXP_ID}.csv"

# Intermediates (job specs, logs, per-run results, checkpoints, the panel parquet) go to
# scratch, NOT to /kaggle/working — otherwise the output browser fills with ~50 files across
# four subfolders and downloading anything becomes painful. /kaggle/temp is not persisted as
# notebook output; §14 packages everything worth keeping into a single zip at the end.
def _scratch_root():
    # /kaggle/temp does not exist on every Kaggle image (it did not in run 2), so fall back
    # to /tmp -- also non-persisted -- before ever polluting the downloadable output dir.
    for cand in (Path("/kaggle/temp"), Path("/tmp/tft_scratch")):
        try:
            cand.mkdir(parents=True, exist_ok=True)
            (cand / ".w").write_text("ok"); (cand / ".w").unlink()
            return cand
        except Exception:
            continue
    return OUTPUT_DIR

SCRATCH_ROOT = _scratch_root()
RUN_DIR    = SCRATCH_ROOT / "run"
# Model weights go to the DOWNLOADABLE output dir, not scratch: the final fit costs ~1 h of
# GPU and is otherwise lost when the session ends.
MODEL_DIR  = OUTPUT_DIR / "models"
PANEL_PATH = RUN_DIR / "panel.parquet"
WORKER     = RUN_DIR / "tft_worker.py"
for d in (RUN_DIR, RUN_DIR / "jobs", RUN_DIR / "logs", RUN_DIR / "results", RUN_DIR / "ckpt"):
    d.mkdir(parents=True, exist_ok=True)

# ── Reproducibility & CV (identical fold boundaries to exp-11) ────────────────
SEED         = 42
SEEDS        = [42, 7, 13, 99, 1234]   # exp-12 shipped ONE seed
N_FOLDS      = 3
EVAL_HORIZON = 336

# ── Features (same 11 domain forecasts + block-C velocities as exp-11) ────────
DOMAIN_FEAT_COLS = [
    "workload_intensity", "demand_forecast", "upstream_quality_forecast",
    "promotion_intensity", "shock_risk", "maintenance_known",
    "unit_reliability_forecast", "queue_pressure_forecast",
    "network_pressure_forecast", "event_load_forecast",
    "service_irregularity_risk_forecast",
]
VELOCITY_COLS  = ["queue_pressure_forecast", "network_pressure_forecast", "shock_risk"]
TIME_REAL_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "doy_sin", "doy_cos", "is_weekend"]
TIME_CAT_COLS  = ["hour_cat", "dow_cat"]

# ── Windowing ─────────────────────────────────────────────────────────────────
MAX_ENCODER      = 336
MIN_ENCODER      = 168          # == MAX_ENCODER disables the crop augmentation
RANDOMIZE_LENGTH = [0.2, 0.05]  # null disables; this is the exp-11 crop analogue
STRIDE           = 24           # one training window per series per day

# ── TFT architecture ──────────────────────────────────────────────────────────
HIDDEN_SIZE             = 96
HIDDEN_CONTINUOUS_SIZE  = 32
ATTENTION_HEADS         = 4
LSTM_LAYERS             = 1
DROPOUT                 = 0.15
CAUSAL_ATTENTION        = False   # False = decoder may attend to known future covariates
SHARE_VARIABLE_NETWORKS = False
LOSS_NAME               = "mae"   # "mae" (WAPE-aligned) | "quantile"

# ── Optimisation ──────────────────────────────────────────────────────────────
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4
OPTIMIZER     = "adamw"
GRAD_CLIP     = 0.5
BATCH_SIZE    = 32
ACCUM_BATCHES = 4                 # effective batch 128
EVAL_BATCH    = 32
MAX_EPOCHS    = 60
PATIENCE      = 10
# §7 only needs to RANK arms, and at real scale each arm is a full-length run. Early stopping
# usually fires well before this anyway.
PROBE_EPOCHS  = 20
PROBE_ARMS_N  = 4       # §7 arm count, named here so §6 can price the probe stage

# 2 concurrent jobs share ~4 vCPU, so keep loader workers low or the CPU becomes
# the bottleneck and both GPUs starve.
N_WORKERS_PER_JOB = 1

# ── Protocol ──────────────────────────────────────────────────────────────────
# "inner": early-stop on the 336h block *before* the fold test block. The test block never
#          influences selection -> honest CV, costs 336h of training data.
# "fold" : early-stop on the fold test block itself — reproduces exp-11's protocol, so
#          numbers are directly comparable to 14.203, but optimistic.
EARLY_STOP_MODE = "inner"

# ── Stage switches ────────────────────────────────────────────────────────────
# Stage 1 runs §11 only. §7/§8 are not repeated — the config and epoch count come from
# exp-12 run 4, and re-running them would cost ~14 GPU-h to reproduce numbers we already have.
RUN_THROUGHPUT_PROBE = True      # 3 min: re-validates fp16, both GPUs, and the premises
RUN_CONFIG_PROBE     = False
RUN_FULL_CV          = False
RUN_FINAL_FIT        = True
RUN_BLEND            = False     # §12 needs a matching OOF; stage 0 handles blending
RUN_TUNING           = False      # §13 hyperparameter search — see the budget note

# Lets §11 run WITHOUT §7/§8. The final fit normally takes its epoch count from the CV median
# best_epoch, which lives in §8's in-memory epoch_log. Set this to that number (run 4: 10) to
# re-run the final fit alone -- e.g. to regenerate a checkpoint -- for ~1 job instead of 13.
FINAL_EPOCHS_OVERRIDE = 10   # CV median best_epoch from exp-12 run 4

# ── PREFLIGHT ─────────────────────────────────────────────────────────────────
# Runs EVERY stage end-to-end in ~5 minutes on the real data, at the real architecture and
# the real precision — only the epoch count, stride and seed count are cut. It exists to
# prove the plumbing before you spend hours: dispatcher, per-GPU pinning, fp16 + mask_bias,
# peak memory at hidden_size=96, parquet/npz round-trips, OOF assembly, submission asserts,
# blend, tuning. It proves nothing about accuracy — the WAPE numbers will be garbage.
#
# Run it once with PREFLIGHT = True. If it completes, set PREFLIGHT = False and run for real.
PREFLIGHT = False

# Real-run values, captured before PREFLIGHT overrides them, so §6 can still project the
# real budget from preflight measurements.
REAL_STRIDE, REAL_MAX_EPOCHS, REAL_SEEDS = STRIDE, MAX_EPOCHS, list(SEEDS)
REAL_PROBE_EPOCHS = PROBE_EPOCHS

# Subset the panel during preflight. This is the single biggest speed lever: TimeSeriesDataSet
# enumerates every window before filtering (221,280 rows per fold at 96 series), and that index
# is rebuilt in every worker process. Per-step GPU cost and memory depend on batch x sequence
# length, NOT on the number of series, so the fp16 / peak-memory numbers stay representative.
PREFLIGHT_SERIES = 16

if PREFLIGHT:
    EXP_ID   = f"{EXP_ID}_preflight"      # never overwrite real results with toy ones
    # TWO seeds, not one: multi-seed averaging IS this experiment, so preflight must
    # actually exercise it. One extra ~1-minute job buys coverage of the thing under test.
    SEEDS    = SEEDS[:2]
    STRIDE   = 168
    MAX_EPOCHS, PATIENCE, PROBE_EPOCHS = 1, 1, 1
    TUNE_TRIALS_OVERRIDE = 2              # §13 is off in stage 1; nothing to exercise
    # architecture, encoder lengths, batch size and precision stay at REAL values so the
    # memory footprint and fp16 behaviour measured here are the ones you will actually hit.
    OOF_OUT        = OUTPUT_DIR / f"oof_predictions_{EXP_ID}.csv"
    VAL_OUT        = OUTPUT_DIR / f"validation_predictions_{EXP_ID}.csv"
    PROBE_OUT      = OUTPUT_DIR / f"probe_{EXP_ID}.csv"
    IMPORTANCE_OUT = OUTPUT_DIR / f"variable_importance_{EXP_ID}.csv"
else:
    PREFLIGHT_SERIES = None
    TUNE_TRIALS_OVERRIDE = None

print(f"config loaded  |  PREFLIGHT={PREFLIGHT}  EXP_ID={EXP_ID}")
print(f"  outputs (downloadable): {OUTPUT_DIR}")
print(f"  intermediates (scratch): {RUN_DIR}")
if PREFLIGHT:
    print("  -> toy epochs/stride/seeds; WAPE numbers are meaningless by design")
    print("  -> architecture, encoder, batch size and precision are at REAL values")

config loaded  |  PREFLIGHT=False  EXP_ID=13a_tft_multiseed
  outputs (downloadable): /kaggle/working
  intermediates (scratch): /kaggle/temp/run


---
## §1. Environment

The CUDA assertion is deliberate: if pip swaps the GPU build of torch for a CPU one, every
timing number below becomes meaningless and you would only find out hours later.

In [2]:
import subprocess, sys, time, warnings
warnings.filterwarnings("ignore")

# Pinned: every API semantic this notebook relies on (loss computed on the original target
# scale, .filter() index columns, randomize_length, mask_bias, predict return shapes) was
# verified against 1.8.0. An unpinned install can drift under you silently.
PF_VERSION = "1.8.0"
try:
    import pytorch_forecasting  # noqa: F401
    assert pytorch_forecasting.__version__ == PF_VERSION
except (ImportError, AssertionError):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
                    f"pytorch-forecasting=={PF_VERSION}"], check=True)
    print("installed pytorch-forecasting — RESTART THE KERNEL if imports below fail")

import lightning.pytorch as pl
import numpy as np
import pandas as pd
import torch
import pytorch_forecasting as pf

print(f"torch               {torch.__version__}")
print(f"lightning           {pl.__version__}")
print(f"pytorch-forecasting {pf.__version__}")
print(f"pandas              {pd.__version__}")

assert torch.cuda.is_available(), "No CUDA device — pip likely replaced the GPU build of torch."
N_GPUS = torch.cuda.device_count()
for i in range(N_GPUS):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
if N_GPUS < 2:
    print("\n!! Only one GPU visible. Set the Kaggle accelerator to 'GPU T4 x2'.")
    print("   The notebook still runs correctly — it will just take ~2x longer.")
print(f"CPU cores: {os.cpu_count()}   concurrent jobs: {N_GPUS}")


def wape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + eps) * 100)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 259.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 268.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 299.5 MB/s eta 0:00:00
installed pytorch-forecasting — RESTART THE KERNEL if imports below fail
torch               2.10.0+cu128
lightning           2.6.5
pytorch-forecasting 1.8.0
pandas              2.3.3
GPU 0: Tesla T4
GPU 1: Tesla T4
CPU cores: 4   concurrent jobs: 2


---
## §2. Data preparation



In [3]:
def impute_domain_features(df, cols):
    df = df.sort_values(["series_id", "timestamp"]).copy()
    df[cols] = df.groupby("series_id", group_keys=False)[cols].transform(lambda x: x.ffill().bfill())
    return df


def add_time_features(df, time_col="timestamp"):
    df = df.copy()
    t = pd.to_datetime(df[time_col])
    df["hour_sin"]   = np.sin(2 * np.pi * t.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * t.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * t.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * t.dt.dayofweek / 7)
    df["doy_sin"]    = np.sin(2 * np.pi * t.dt.dayofyear / 365)
    df["doy_cos"]    = np.cos(2 * np.pi * t.dt.dayofyear / 365)
    df["is_weekend"] = (t.dt.dayofweek >= 5).astype(float)
    df["hour_cat"]   = t.dt.hour.astype(str)
    df["dow_cat"]    = t.dt.dayofweek.astype(str)
    return df


train_df  = pd.read_csv(TRAIN_PATH,     parse_dates=["timestamp"])
val_input = pd.read_csv(VAL_INPUT_PATH, parse_dates=["timestamp"])
val_index = pd.read_csv(VAL_INDEX_PATH, parse_dates=["timestamp"])

N_SERIES_FULL = train_df["series_id"].nunique()
if PREFLIGHT and PREFLIGHT_SERIES and PREFLIGHT_SERIES < N_SERIES_FULL:
    keep = sorted(train_df["series_id"].unique())[:PREFLIGHT_SERIES]
    train_df  = train_df[train_df.series_id.isin(keep)].copy()
    val_input = val_input[val_input.series_id.isin(keep)].copy()
    val_index = val_index[val_index.series_id.isin(keep)].copy()
    print(f"PREFLIGHT: {len(keep)} of {N_SERIES_FULL} series "
          f"(per-step cost and memory are unchanged; only the window index shrinks)")

train_df  = add_time_features(impute_domain_features(train_df,  DOMAIN_FEAT_COLS))
val_input = add_time_features(impute_domain_features(val_input, DOMAIN_FEAT_COLS))

train_df["is_future"]  = False
val_input["is_future"] = True
val_input["target"]    = 0.0          # dummy; never used as a label

panel = pd.concat([train_df, val_input], ignore_index=True)
panel = panel.sort_values(["series_id", "timestamp"]).reset_index(drop=True)

T0 = panel["timestamp"].min()
panel["time_idx"] = ((panel["timestamp"] - T0).dt.total_seconds() // 3600).astype(int)

for c in VELOCITY_COLS:                                   # block-C velocities
    panel[f"{c}_vel"] = panel.groupby("series_id")[c].diff().fillna(0.0)
VEL_COLS = [f"{c}_vel" for c in VELOCITY_COLS]

panel["series_id"] = panel["series_id"].astype(str)
panel["target"]    = panel["target"].astype(np.float32)

N_SERIES = panel["series_id"].nunique()
T_TRAIN  = int(panel.loc[~panel.is_future, "time_idx"].max()) + 1
T_TOTAL  = int(panel["time_idx"].max()) + 1

print(f"panel      : {panel.shape}")
print(f"series     : {N_SERIES}")
print(f"train hours: {T_TRAIN}   forecast hours: {T_TOTAL - T_TRAIN}")

counts = panel.groupby("series_id").size()
assert counts.nunique() == 1 and counts.iloc[0] == T_TOTAL, f"ragged panel:\n{counts.describe()}"
gaps = panel.groupby("series_id")["time_idx"].apply(lambda s: s.diff().dropna().ne(1).sum())
assert gaps.sum() == 0, "non-contiguous hourly grid — TimeSeriesDataSet needs allow_missing_timesteps"
assert panel[DOMAIN_FEAT_COLS].isna().sum().sum() == 0, "NaNs survived imputation"
assert panel.loc[~panel.is_future, "target"].notna().all(), "NaN targets in the training block"
print("integrity checks passed")

panel.to_parquet(PANEL_PATH, index=False)
print(f"wrote {PANEL_PATH}")

panel      : (446976, 34)
series     : 96
train hours: 4320   forecast hours: 336
integrity checks passed
wrote /kaggle/temp/run/panel.parquet


---
## §3. CV framework & the shared job config

Fold boundaries are **identical to exp-11**, so CV numbers sit on the same scale:

| fold | train | test |
|---|---|---|
| 0 | [0..3312) | [3312..3648) |
| 1 | [0..3648) | [3648..3984) |
| 2 | [0..3984) | [3984..4320) |


In [4]:
def fold_train_end(k):
    """Last exclusive time_idx of fold k's training slice."""
    return T_TRAIN - (N_FOLDS - k) * EVAL_HORIZON


def fold_test_slice(k):
    s = fold_train_end(k)
    return s, s + EVAL_HORIZON


for k in range(N_FOLDS):
    a, b = fold_test_slice(k)
    print(f"fold {k}: train=[0..{fold_train_end(k)})  test=[{a}..{b})")

KNOWN_REALS   = ["time_idx"] + DOMAIN_FEAT_COLS + VEL_COLS + TIME_REAL_COLS
UNKNOWN_REALS = ["target"]

CFG = dict(
    panel_path=str(PANEL_PATH),
    ckpt_dir=str(RUN_DIR / "ckpt"),
    model_dir=str(MODEL_DIR),
    results_dir=str(RUN_DIR / "results"),
    eval_horizon=EVAL_HORIZON,
    n_series=int(N_SERIES),
    t_train=int(T_TRAIN),
    static_categoricals=["series_id"],
    time_varying_known_categoricals=TIME_CAT_COLS,
    time_varying_known_reals=KNOWN_REALS,
    time_varying_unknown_reals=UNKNOWN_REALS,
    min_encoder=MIN_ENCODER, max_encoder=MAX_ENCODER,
    randomize_length=RANDOMIZE_LENGTH, stride=STRIDE,
    hidden_size=HIDDEN_SIZE, hidden_continuous_size=HIDDEN_CONTINUOUS_SIZE,
    attention_heads=ATTENTION_HEADS, lstm_layers=LSTM_LAYERS, dropout=DROPOUT,
    causal=CAUSAL_ATTENTION, share_vsn=SHARE_VARIABLE_NETWORKS, loss_name=LOSS_NAME,
    learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, optimizer=OPTIMIZER,
    grad_clip=GRAD_CLIP, batch_size=BATCH_SIZE, accum=ACCUM_BATCHES,
    eval_batch=EVAL_BATCH, max_epochs=MAX_EPOCHS, patience=PATIENCE,
    n_workers=N_WORKERS_PER_JOB,
)


def make_job(tag, *, kind="train", train_end, eval_start, es_start=None, seed=SEED,
             fixed_epochs=None, limit_train_batches=None, save_ckpt=False, **overrides):
    """One unit of GPU work. `overrides` patch CFG for this job only (e.g. causal=True)."""
    cfg = dict(CFG); cfg.update(overrides)
    return dict(tag=tag, kind=kind, cfg=cfg, seed=seed,
                train_end=int(train_end), eval_start=int(eval_start),
                es_start=None if es_start is None else int(es_start),
                fixed_epochs=fixed_epochs, limit_train_batches=limit_train_batches,
                save_ckpt=save_ckpt)


def fold_bounds(k):
    """(train_end, es_start, oof_start) for fold k under the configured protocol."""
    end = fold_train_end(k)
    if EARLY_STOP_MODE == "inner":
        return end - EVAL_HORIZON, end - EVAL_HORIZON, end
    if EARLY_STOP_MODE == "fold":
        return end, end, end
    raise ValueError(EARLY_STOP_MODE)


print(f"\nearly-stop mode: {EARLY_STOP_MODE}")
for k in range(N_FOLDS):
    te, es, oof = fold_bounds(k)
    print(f"  fold {k}: train=[0..{te})  early-stop=[{es}..{es+EVAL_HORIZON})  score=[{oof}..{oof+EVAL_HORIZON})")

fold 0: train=[0..3312)  test=[3312..3648)
fold 1: train=[0..3648)  test=[3648..3984)
fold 2: train=[0..3984)  test=[3984..4320)

early-stop mode: inner
  fold 0: train=[0..2976)  early-stop=[2976..3312)  score=[3312..3648)
  fold 1: train=[0..3312)  early-stop=[3312..3648)  score=[3648..3984)
  fold 2: train=[0..3648)  early-stop=[3648..3984)  score=[3984..4320)


---
## §4. The worker

One self-contained script, one job, one GPU. Written to disk here and launched as a
subprocess by the dispatcher in §5 with `CUDA_VISIBLE_DEVICES` pinned, so each worker sees
exactly one device as `cuda:0`.

Every leakage-critical invariant is asserted **inside the worker**, so a misconfigured job
fails loudly on that GPU instead of silently producing an optimistic number.

In [5]:
worker_src = r"""
import argparse, json, shutil, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")
from lightning.pytorch import Trainer, seed_everything
from lightning.pytorch.callbacks import Callback, EarlyStopping, ModelCheckpoint
from pytorch_forecasting import GroupNormalizer, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import MAE, QuantileLoss


def wape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + eps) * 100)


class StepTimer(Callback):
    # Per-training-step wall time. CUDA is async, so synchronise or the numbers lie.
    # Only attached to warmup jobs -- syncing every step would tax real training.

    def __init__(self):
        self.times, self._t = [], None

    def on_train_batch_start(self, *a, **k):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self._t = time.perf_counter()

    def on_train_batch_end(self, *a, **k):
        if self._t is None:
            return
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.times.append(time.perf_counter() - self._t)


class EpochLog(Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        vl = trainer.callback_metrics.get("val_loss")
        print(f"    epoch {trainer.current_epoch:3d}  val_loss {float(vl):.4f}", flush=True)


def build_train_ds(panel, cfg, train_end):
    min_enc, max_enc = cfg["min_encoder"], cfg["max_encoder"]
    randomize = cfg["randomize_length"]
    if min_enc == max_enc:
        randomize = None
    if randomize is not None:
        randomize = tuple(randomize)
    ds = TimeSeriesDataSet(
        panel[panel.time_idx < train_end],
        time_idx="time_idx", target="target", group_ids=["series_id"],
        min_encoder_length=min_enc, max_encoder_length=max_enc,
        min_prediction_length=cfg["eval_horizon"], max_prediction_length=cfg["eval_horizon"],
        static_categoricals=cfg["static_categoricals"],
        time_varying_known_categoricals=cfg["time_varying_known_categoricals"],
        time_varying_known_reals=cfg["time_varying_known_reals"],
        time_varying_unknown_reals=cfg["time_varying_unknown_reals"],
        target_normalizer=GroupNormalizer(groups=["series_id"], transformation=None),
        add_relative_time_idx=True, add_target_scales=True,
        add_encoder_length=(min_enc != max_enc),
        randomize_length=randomize, allow_missing_timesteps=False,
    )
    n_all = len(ds)
    stride = cfg["stride"]
    if stride and stride > 1:
        # Anchor the stride grid to the LAST valid window, not to absolute time 0. Anchoring
        # at 0 keeps windows where time_idx % stride == 0, which drops the final window
        # whenever stride does not divide max_start -- the model then never trains on windows
        # reaching the end of history, which is exactly exp-11's Bug 1 (LB 17.9 -> 14.8).
        # Stepping backwards from max_start makes the last window survive for ANY stride.
        # Also DEDUPLICATE. When min_encoder_length < max_encoder_length, pytorch-forecasting
        # emits one index row per allowed encoder length -- ~12 rows per decoder start -- so
        # filtering on time_idx_first_prediction alone keeps every variant and inflates the
        # dataset ~12x (measured). randomize_length already crops the encoder per sample at
        # __getitem__ time, so keep ONE row per start (the longest) and let the augmentation
        # do the cropping. Without this the stride does not mean what it says, arms with
        # different min_encoder are not compute-comparable, and the budget blows up.
        di = ds.decoded_index.reset_index(drop=True)
        seq = ds.index["sequence_length"].to_numpy()
        max_start = int(di.time_idx_first_prediction.max())
        keyed = pd.DataFrame({"g": di["series_id"].astype(str),
                              "t": di["time_idx_first_prediction"].to_numpy(),
                              "s": seq})
        longest = keyed.groupby(["g", "t"])["s"].transform("max").to_numpy() == seq
        on_grid = ((max_start - keyed["t"].to_numpy()) % stride) == 0
        sel = keyed.loc[longest & on_grid]
        keep = np.zeros(len(keyed), dtype=bool)
        keep[sel.index[~sel.duplicated(["g", "t"])]] = True
        ds = ds.filter(lambda _idx: keep)
    print(f"  train ds [0..{train_end}) {n_all:,} windows -> {len(ds):,} at stride {stride}", flush=True)

    # the last training window must reach the end of the available history
    di = ds.decoded_index
    assert int(di.time_idx_first_prediction.max()) + cfg["eval_horizon"] == train_end, (
        f"last training window ends at "
        f"{int(di.time_idx_first_prediction.max()) + cfg['eval_horizon']}, not {train_end}")

    last_target = int((ds.decoded_index.time_idx_first_prediction + cfg["eval_horizon"] - 1).max())
    assert last_target < train_end, f"LEAK: training targets reach {last_target} >= {train_end}"
    return ds


def build_eval_ds(train_ds, panel, cfg, decoder_start):
    ds = TimeSeriesDataSet.from_dataset(
        train_ds, panel[panel.time_idx < decoder_start + cfg["eval_horizon"]],
        predict=True, stop_randomization=True)
    starts = ds.decoded_index.time_idx_first_prediction.unique()
    assert len(starts) == 1 and starts[0] == decoder_start, f"window misaligned: {starts} != {decoder_start}"
    assert len(ds) == cfg["n_series"], f"expected {cfg['n_series']} windows, got {len(ds)}"
    return ds


def to_dl(ds, cfg, *, train):
    return ds.to_dataloader(
        train=train, batch_size=cfg["batch_size"] if train else cfg["eval_batch"],
        num_workers=cfg["n_workers"], pin_memory=True,
        persistent_workers=cfg["n_workers"] > 0)


def build_model(train_ds, cfg, precision, causal=None):
    mask_bias = -float("inf") if precision != "32-true" else -1e9
    return TemporalFusionTransformer.from_dataset(
        train_ds,
        hidden_size=cfg["hidden_size"], hidden_continuous_size=cfg["hidden_continuous_size"],
        attention_head_size=cfg["attention_heads"], lstm_layers=cfg["lstm_layers"],
        dropout=cfg["dropout"],
        causal_attention=cfg["causal"] if causal is None else causal,
        share_single_variable_networks=cfg["share_vsn"],
        loss=MAE() if cfg["loss_name"] == "mae" else QuantileLoss(),
        learning_rate=cfg["learning_rate"],
        optimizer=cfg["optimizer"],
        weight_decay=cfg["weight_decay"],   # top-level kwarg; via optimizer_params it raises TypeError
        mask_bias=mask_bias, log_interval=-1, reduce_on_plateau_patience=4,
    )


def main(job_path):
    job = json.loads(Path(job_path).read_text())
    cfg, tag = job["cfg"], job["tag"]
    t_start = time.perf_counter()
    wall_start = time.time()

    # bf16 needs Ampere+; a T4 is Turing -> fp16.
    precision = "bf16-mixed" if torch.cuda.is_bf16_supported() else "16-mixed"
    print(f"[{tag}] device={torch.cuda.get_device_name(0)} precision={precision}", flush=True)

    panel = pd.read_parquet(cfg["panel_path"])
    train_ds = build_train_ds(panel, cfg, job["train_end"])
    eval_ds  = build_eval_ds(train_ds, panel, cfg, job["eval_start"])
    es_ds    = None if job["es_start"] is None else build_eval_ds(train_ds, panel, cfg, job["es_start"])

    seed_everything(job["seed"], workers=True)
    model = build_model(train_ds, cfg, precision)
    n_params = sum(p.numel() for p in model.parameters())
    train_dl = to_dl(train_ds, cfg, train=True)
    steps_per_epoch = len(train_dl)

    callbacks, ckpt = [EpochLog()], None
    # warmup jobs (the only ones with limit_train_batches) get precise step timing
    timer = StepTimer() if job["limit_train_batches"] else None
    if timer is not None:
        callbacks.append(timer)
    fixed = job["fixed_epochs"]
    if fixed is None:
        assert es_ds is not None, "early stopping requested without an early-stop window"
        ckpt = ModelCheckpoint(dirpath=cfg["ckpt_dir"], filename=tag,
                               monitor="val_loss", mode="min", save_top_k=1)
        callbacks += [ckpt, EarlyStopping(monitor="val_loss", min_delta=1e-4,
                                          patience=cfg["patience"], mode="min")]

    trainer = Trainer(
        max_epochs=fixed or cfg["max_epochs"],
        accelerator="gpu", devices=1, precision=precision,
        gradient_clip_val=cfg["grad_clip"], accumulate_grad_batches=cfg["accum"],
        limit_train_batches=job["limit_train_batches"] or 1.0,
        callbacks=callbacks, logger=False,
        enable_progress_bar=False, enable_model_summary=False,
        enable_checkpointing=ckpt is not None,
    )
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    fit_t0 = time.perf_counter()
    trainer.fit(model, train_dl, None if fixed is not None else to_dl(es_ds, cfg, train=False))
    fit_secs = time.perf_counter() - fit_t0

    best_epoch, best_score = trainer.current_epoch, float("nan")
    if ckpt is not None and ckpt.best_model_path:
        model = TemporalFusionTransformer.load_from_checkpoint(ckpt.best_model_path)
        best_epoch = max(trainer.current_epoch - cfg["patience"], 1)
        best_score = float(ckpt.best_model_score)
        if not job["save_ckpt"]:
            Path(ckpt.best_model_path).unlink(missing_ok=True)

    # Persist the model when asked. Two distinct cases:
    #  - early-stopped jobs: copy the BEST checkpoint (trainer state holds the LAST epoch,
    #    so trainer.save_checkpoint here would save the wrong weights)
    #  - fixed-epoch jobs (the final fit): there is no val loop, so ModelCheckpoint never
    #    exists and nothing is written unless we do it explicitly
    if job["save_ckpt"]:
        dest = Path(cfg["model_dir"]); dest.mkdir(parents=True, exist_ok=True)
        target = dest / f"{tag}.ckpt"
        if ckpt is not None and ckpt.best_model_path:
            shutil.copy2(ckpt.best_model_path, target)
        else:
            trainer.save_checkpoint(str(target))
        print(f"  saved model -> {target} ({target.stat().st_size/2**20:.1f} MB)", flush=True)

    out = model.predict(
        to_dl(eval_ds, cfg, train=False), mode="prediction", return_index=True,
        trainer_kwargs=dict(accelerator="gpu", devices=1, precision=precision,
                            logger=False, enable_progress_bar=False))
    preds = out.output.cpu().float().numpy()
    index = out.index

    di = train_ds.decoded_index
    res = dict(tag=tag, kind=job["kind"], seed=job["seed"], n_params=int(n_params),
               steps_per_epoch=int(steps_per_epoch), epochs_run=int(trainer.current_epoch),
               best_epoch=int(best_epoch), best_val_loss=best_score,
               fit_seconds=fit_secs, total_seconds=time.perf_counter() - t_start,
               started_at=wall_start, finished_at=time.time(),
               sec_per_epoch=fit_secs / max(trainer.current_epoch, 1),
               eval_start=job["eval_start"], train_end=job["train_end"],
               n_train_windows=int(len(train_ds)),
               checkpoint=(str(Path(cfg["model_dir"]) / f"{tag}.ckpt")
                           if job["save_ckpt"] else None),
               batch_size=int(cfg["batch_size"]),
               peak_gpu_gib=(torch.cuda.max_memory_allocated() / 2**30
                             if torch.cuda.is_available() else 0.0),
               ms_per_step=(float(np.median(timer.times)) * 1000
                            if timer is not None and timer.times else None),
               timed_steps=(len(timer.times) if timer is not None else 0),
               last_train_target=int((di.time_idx_first_prediction + cfg["eval_horizon"] - 1).max()),
               last_train_encoder_end=int(di.time_idx_first_prediction.max()))

    # score only when the evaluated block has real targets
    if job["eval_start"] + cfg["eval_horizon"] <= cfg["t_train"]:
        h = preds.shape[1]
        rows = []
        for i in range(len(index)):
            t0 = int(index["time_idx"].iloc[i])
            rows.append(pd.DataFrame({"series_id": index["series_id"].iloc[i],
                                      "time_idx": np.arange(t0, t0 + h),
                                      "pred": preds[i]}))
        long = pd.concat(rows, ignore_index=True).merge(
            panel[["series_id", "time_idx", "target"]], on=["series_id", "time_idx"], how="left")
        assert long["target"].notna().all(), "missing targets in the scored window"
        res["wape"] = wape(long["target"], long["pred"])

    if job["kind"] == "throughput":
        res.update(sanity_checks(model, trainer, eval_ds, panel, cfg, precision))

    np.savez_compressed(Path(cfg["results_dir"]) / f"{tag}.npz",
                        preds=preds, series_id=index["series_id"].to_numpy().astype(str),
                        time_idx=index["time_idx"].to_numpy().astype(np.int64))
    Path(cfg["results_dir"], f"{tag}.json").write_text(json.dumps(res, indent=1))
    print(f"[{tag}] done in {res['total_seconds']/60:.1f} min"
          + (f" | WAPE {res['wape']:.3f}" if "wape" in res else ""), flush=True)


def sanity_checks(model, trainer, eval_ds, panel, cfg, precision):
    # The two premises this experiment rests on. Both must hold or the rationale is void.
    out = model.predict(to_dl(eval_ds, cfg, train=False), mode="prediction", return_index=True,
                        trainer_kwargs=dict(accelerator="gpu", devices=1, precision=precision,
                                            logger=False, enable_progress_bar=False))
    preds, index, h = out.output.cpu().float().numpy(), out.index, cfg["eval_horizon"]
    rows = []
    for i in range(len(index)):
        t0 = int(index["time_idx"].iloc[i])
        rows.append(pd.DataFrame({"series_id": index["series_id"].iloc[i],
                                  "time_idx": np.arange(t0, t0 + h), "pred": preds[i]}))
    long = pd.concat(rows, ignore_index=True).merge(
        panel[["series_id", "time_idx", "target"]], on=["series_id", "time_idx"], how="left")

    err = (long["pred"] - long["target"]).abs()
    mae_original = float(err.mean())
    scale = panel[panel.time_idx < cfg["t_train"]].groupby("series_id")["target"].std()
    mae_normalized = float((err / long["series_id"].map(scale)).mean())
    reported = float(trainer.validate(model, to_dl(eval_ds, cfg, train=False), verbose=False)[0]["val_loss"])
    on_original = abs(reported - mae_original) < abs(reported - mae_normalized)
    print(f"  loss scale: original {mae_original:.4f} | normalized {mae_normalized:.4f} "
          f"| reported {reported:.4f} -> {'ORIGINAL' if on_original else 'NORMALIZED'}", flush=True)
    assert on_original, "Loss is NOT on the original scale — the WAPE-alignment premise fails."

    batch = next(iter(to_dl(eval_ds, cfg, train=False)))[0]
    batch = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in batch.items()}
    model.eval()
    with torch.no_grad():
        model.hparams.causal_attention = True
        a = model(batch)["prediction"].clone()
        model.hparams.causal_attention = False
        b = model(batch)["prediction"].clone()
    delta = float((a - b).abs().mean())
    print(f"  mean |causal - non-causal| prediction delta: {delta:.5f}", flush=True)
    assert delta > 0, "causal_attention is a no-op — check the library version."
    model.hparams.causal_attention = cfg["causal"]

    return dict(loss_on_original_scale=bool(on_original), mae_original=mae_original,
                mae_normalized=mae_normalized, reported_val_loss=reported,
                causal_delta=delta)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("job")
    main(ap.parse_args().job)
"""

WORKER.write_text(worker_src)
print(f"wrote {WORKER} ({len(worker_src.splitlines())} lines)")

wrote /kaggle/temp/run/tft_worker.py (313 lines)


---
## §5. Dispatcher — one job per GPU, `N_GPUS` at a time

Jobs are queued and launched as subprocesses with `CUDA_VISIBLE_DEVICES` pinned. A GPU
becomes free the moment its job exits, and the next queued job starts immediately, so both
T4s stay busy even when runs finish at different times (early stopping guarantees they will).

If a job fails, the tail of its log is printed and the other GPU keeps working.

In [6]:
JOB_DIR, LOG_DIR, RES_DIR = RUN_DIR / "jobs", RUN_DIR / "logs", RUN_DIR / "results"


def run_jobs(jobs, n_gpus=None, poll=5.0, verbose=True, allow_failures=False):
    """Run jobs across the visible GPUs, one per GPU at a time. -> {tag: result dict}

    allow_failures=True returns the successful results instead of raising. Used by the
    batch-size sweep in §6, where an OOM at the largest batch is a *result*, not a crash.
    """
    n_gpus = n_gpus or N_GPUS
    pending, running, results, failed = list(jobs), {}, {}, []
    free = list(range(n_gpus))
    t_wall = time.perf_counter()

    while pending or running:
        while pending and free:
            job = pending.pop(0)
            gpu = free.pop(0)
            jp = JOB_DIR / f"{job['tag']}.json"
            jp.write_text(json.dumps(job))
            log = LOG_DIR / f"{job['tag']}.log"
            fh = log.open("w")
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
            proc = subprocess.Popen([sys.executable, str(WORKER), str(jp)],
                                    env=env, stdout=fh, stderr=subprocess.STDOUT)
            running[job["tag"]] = (proc, gpu, time.perf_counter(), log, fh)
            if verbose:
                print(f"[dispatch] {job['tag']:<28} -> GPU {gpu}   "
                      f"({len(pending)} queued, {len(running)} running)")

        time.sleep(poll)

        for tag, (proc, gpu, t0, log, fh) in list(running.items()):
            if proc.poll() is None:
                continue
            fh.close()
            del running[tag]
            free.append(gpu)
            mins = (time.perf_counter() - t0) / 60
            rp = RES_DIR / f"{tag}.json"
            if proc.returncode == 0 and rp.exists():
                res = json.loads(rp.read_text())
                results[tag] = res
                extra = f" | WAPE {res['wape']:.3f}" if "wape" in res else ""
                print(f"[done]     {tag:<28} GPU {gpu}  {mins:5.1f} min{extra}")
            else:
                failed.append(tag)
                print(f"[FAILED]   {tag:<28} GPU {gpu}  {mins:5.1f} min  (exit {proc.returncode})")
                print("           " + "\n           ".join(log.read_text().splitlines()[-15:]))

    print(f"\nwall clock: {(time.perf_counter()-t_wall)/60:.1f} min  "
          f"| {len(results)} ok, {len(failed)} failed")
    if failed:
        raise RuntimeError(f"jobs failed: {failed}  (full logs in {LOG_DIR})")
    return results


def load_preds(tag):
    """-> (preds (n_series, horizon), index df[series_id, time_idx])"""
    z = np.load(RES_DIR / f"{tag}.npz", allow_pickle=False)
    return z["preds"], pd.DataFrame({"series_id": z["series_id"].astype(str),
                                     "time_idx": z["time_idx"]})


def preds_to_frame(preds, index, col="pred_tft"):
    """Wide (n_series, horizon) -> long frame joined to timestamp + target."""
    h = preds.shape[1]
    frames = []
    for i in range(len(index)):
        t0 = int(index["time_idx"].iloc[i])
        frames.append(pd.DataFrame({
            "series_id": index["series_id"].iloc[i],
            "time_idx": np.arange(t0, t0 + h),
            col: preds[i].astype(np.float64),
            "lead_h": np.arange(1, h + 1),
        }))
    long = pd.concat(frames, ignore_index=True)
    return long.merge(panel[["series_id", "time_idx", "timestamp", "target"]],
                      on=["series_id", "time_idx"], how="left")


PRIOR_SEARCH_ROOTS = [Path("/kaggle/input"), Path("kaggle_priors"), Path("..")]


def find_prior(filename):
    """Locate a prior-experiment CSV wherever it was attached.

    Kaggle mounts a dataset at /kaggle/input/<slug>/..., and the slug and folder layout
    depend on how the dataset was created -- hardcoding /kaggle/input/exp11/... silently
    skipped the residual-correlation gate in run 1. Search by filename instead.
    """
    for root in PRIOR_SEARCH_ROOTS:
        try:
            if not root.exists():
                continue
            hits = sorted(root.rglob(filename))
            if hits:
                return hits[0]
        except (PermissionError, OSError):
            continue
    return None


def pick_pred_col(df, path):
    """Choose the prediction column, refusing to guess when it is ambiguous.

    `oof_predictions_7b_blend.csv` carries pred_lgbm, pred_mlp, pred_seq AND pred_blend;
    taking the first match silently compared against a *component* instead of the shipped
    blend.
    """
    cands = [c for c in df.columns if c.startswith("pred")]
    for preferred in ("pred_blend", "prediction", "pred"):
        if preferred in cands:
            return preferred
    if len(cands) == 1:
        return cands[0]
    raise ValueError(f"{Path(path).name}: ambiguous prediction columns {cands} — "
                     f"name the intended one explicitly")


print("dispatcher ready")

dispatcher ready


---
## §6. Throughput probe + premise checks — the budget gate

**Run this first and read the projection.** Two identical short jobs are launched, one per
GPU, so the measured `sec/epoch` already includes the CPU contention of running both at once
— which is the number that actually prices the rest of the notebook.

The same job also asserts the two premises the experiment rests on: that the loss is computed
on the **original** target scale (if not, TFT inherits exp-11's objective/metric mismatch and
reason #1 evaporates), and that `causal_attention` genuinely changes the forward pass.

Budget: 4 probe arms + 3 folds × 3 seeds = 13 runs, ÷2 GPUs ≈ 7 waves. Kaggle allows 30
GPU-h/week and 12h per session. **If a wave is projected above ~1.5 h, cut `SEEDS` to `[42]`
before §8** — a single-seed 3-fold CV still answers the ship/no-ship question.

In [7]:
# TFT does many small matmuls over 672 timesteps, so it tends to be launch/memory-bound
# rather than FLOP-bound -- a bigger batch is often nearly free per sample. Measured, not
# assumed: this sweeps batch sizes and reports ms/sample and peak memory for each.
BATCH_PROBE = [32, 64, 128]

if RUN_THROUGHPUT_PROBE:
    te, es, _ = fold_bounds(0)
    # Warmup uses the REAL stride so there are enough windows to time properly; the cost is
    # capped by limit_train_batches, so a denser index costs nothing but index construction.
    probe_jobs = [
        make_job(f"warmup_bs{bs}", kind=("throughput" if j == 0 else "train"),
                 train_end=te, es_start=es, eval_start=es, seed=SEED,
                 limit_train_batches=25, max_epochs=1, patience=1,
                 batch_size=bs, stride=REAL_STRIDE)
        for j, bs in enumerate(BATCH_PROBE)
    ]
    # An OOM at the largest batch is information, not a crash — keep going.
    warm = run_jobs(probe_jobs, allow_failures=True)
    assert probe_jobs[0]["tag"] in warm, (
        f"the baseline warmup at batch {BATCH_PROBE[0]} failed — see {LOG_DIR}")

    rows = []
    for bs in BATCH_PROBE:
        r = warm.get(f"warmup_bs{bs}")
        if r is None:
            rows.append(dict(batch=bs, ms_per_step=np.nan, ms_per_sample=np.nan,
                             peak_gib=np.nan, timed_steps=0, note="FAILED (likely OOM)"))
            continue
        # measured by StepTimer with CUDA sync, so validation overhead is excluded
        ms_step = r.get("ms_per_step")
        if ms_step is None:
            ms_step = r["sec_per_epoch"] / max(min(25, r["steps_per_epoch"]), 1) * 1000
        rows.append(dict(batch=bs, ms_per_step=round(ms_step, 1),
                         ms_per_sample=round(ms_step / bs, 2),
                         peak_gib=round(r.get("peak_gpu_gib", 0.0), 2),
                         timed_steps=r.get("timed_steps", 0), note=""))
    bt = pd.DataFrame(rows)
    display(bt)
    if bt["timed_steps"].min() < 5:
        print("!! Few timed steps — treat ms/sample as rough.")

    r = warm[probe_jobs[0]["tag"]]
    ok = bt.dropna(subset=["ms_per_sample"])
    best = ok.loc[ok["ms_per_sample"].idxmin()]
    cfg_row = ok[ok["batch"] == BATCH_SIZE]
    ms_per_sample = float(cfg_row["ms_per_sample"].iloc[0]) if len(cfg_row) else float(best["ms_per_sample"])
    print(f"params          : {r['n_params']:,}")
    print(f"train windows   : {r['n_train_windows']:,} "
          f"({'preflight subset' if PREFLIGHT else 'full dataset'}, fold 0)")
    print(f"configured batch: {BATCH_SIZE} at {ms_per_sample:.2f} ms/sample")
    print(f"fastest batch   : {int(best['batch'])} at {best['ms_per_sample']:.2f} ms/sample, "
          f"peak {best['peak_gib']:.2f} GiB of ~15 GiB")
    if int(best["batch"]) != BATCH_SIZE:
        print(f"  -> set BATCH_SIZE = {int(best['batch'])} in §0 and re-run from §3 for a "
              f"{ms_per_sample/float(best['ms_per_sample']):.2f}x speedup "
              f"(the projection below uses the CONFIGURED batch, not the fastest)")
    print(f"loss on original scale: {r['loss_on_original_scale']}  "
          f"(orig {r['mae_original']:.4f} vs norm {r['mae_normalized']:.4f}, reported {r['reported_val_loss']:.4f})")
    print(f"causal/non-causal delta: {r['causal_delta']:.5f}")

    # Project the REAL run. ms/step is measured at the real architecture so it transfers;
    # the window count does not (preflight subsets series and widens the stride), so
    # recompute that analytically from the pre-override values.
    # Windows start at MIN_ENCODER (not MAX_ENCODER) and folds 1-2 train on more history,
    # so average across folds -- projecting from fold 0 alone understates by ~20%.
    per_fold = [N_SERIES_FULL * ((fold_bounds(k)[0] - EVAL_HORIZON - MIN_ENCODER)
                                 // REAL_STRIDE + 1) for k in range(N_FOLDS)]
    real_windows = int(np.mean(per_fold))
    real_steps = int(np.ceil(real_windows / BATCH_SIZE))
    real_sec_per_epoch = real_windows * ms_per_sample / 1000

    print(f"\n--- projected REAL run ({N_SERIES_FULL} series, stride {REAL_STRIDE}) ---")
    print(f"windows per fold: {per_fold} -> mean {real_windows:,}")
    print(f"{real_steps} steps/epoch @ batch {BATCH_SIZE} "
          f"-> {real_sec_per_epoch/60:.2f} min/epoch")

    n_cv = N_FOLDS * len(REAL_SEEDS)
    # price only the enabled stages, or the projection warns about work that will not run
    stages = []
    if RUN_CONFIG_PROBE:
        stages.append(dict(stage="§7 probe", runs=PROBE_ARMS_N, epochs=REAL_PROBE_EPOCHS))
    if RUN_FULL_CV:
        stages.append(dict(stage="§8 CV", runs=n_cv, epochs=REAL_MAX_EPOCHS))
    if RUN_FINAL_FIT:
        stages.append(dict(stage="§11 final", runs=len(REAL_SEEDS),
                           epochs=FINAL_EPOCHS_OVERRIDE or REAL_MAX_EPOCHS // 2))
    plan = pd.DataFrame(stages) if stages else pd.DataFrame(
        [dict(stage="(nothing enabled)", runs=0, epochs=0)])
    plan["gpu_h"] = (plan.runs * plan.epochs * real_sec_per_epoch / 3600).round(2)
    plan["waves"] = np.ceil(plan.runs / max(N_GPUS, 1)).astype(int)
    plan["wall_h"] = (plan.waves * plan.epochs * real_sec_per_epoch / 3600).round(2)
    display(plan)
    total_gpu_h, total_wall_h = plan.gpu_h.sum(), plan.wall_h.sum()
    print(f"TOTAL: {total_gpu_h:.1f} GPU-hours, ~{total_wall_h:.1f} h wall clock "
          f"(quota 30 GPU-h/week; a session caps at 12 h)")
    if total_gpu_h > 30 or total_wall_h > 12:
        # Cost BOTH constraints for each candidate. Quota and the session cap bind
        # differently -- fewer seeds cuts GPU-hours but barely moves wall clock, because
        # wall clock is driven by waves x epochs.
        def cost(seeds, epochs, probe_ep):
            waves = lambda n: int(np.ceil(n / max(N_GPUS, 1)))
            h = real_sec_per_epoch / 3600
            gpu = (PROBE_ARMS_N * probe_ep + N_FOLDS * seeds * epochs
                   + seeds * (epochs // 2)) * h
            wall = (waves(PROBE_ARMS_N) * probe_ep + waves(N_FOLDS * seeds) * epochs
                    + waves(seeds) * (epochs // 2)) * h
            return gpu, wall

        half = max(REAL_MAX_EPOCHS // 2, 1)
        opts = [(len(REAL_SEEDS), REAL_MAX_EPOCHS, REAL_PROBE_EPOCHS),
                (1, REAL_MAX_EPOCHS, REAL_PROBE_EPOCHS),
                (1, half, REAL_PROBE_EPOCHS),
                (1, half, max(REAL_PROBE_EPOCHS // 2, 5))]
        rows = []
        for s, e, pe in opts:
            gpu, wall = cost(s, e, pe)
            rows.append(dict(SEEDS=s, MAX_EPOCHS=e, PROBE_EPOCHS=pe,
                             gpu_h=round(gpu, 1), wall_h=round(wall, 1),
                             fits=("YES" if gpu <= 30 and wall <= 12 else "no")))
        print("\n" + "!" * 70)
        print("STOP: this does not fit. Change §0 before running §7/§8.")
        print("Both limits must hold: GPU-h <= 30 (weekly quota), wall_h <= 12 (session cap).")
        display(pd.DataFrame(rows))
        ok = [r for r in rows if r["fits"] == "YES"]
        if ok:
            b = ok[0]
            print(f"Smallest change that fits: SEEDS=[{SEED}], MAX_EPOCHS={b['MAX_EPOCHS']}, "
                  f"PROBE_EPOCHS={b['PROBE_EPOCHS']}  ->  {b['gpu_h']} GPU-h / {b['wall_h']} h")
        else:
            print("None of the above fit — also raise STRIDE (24 -> 32/48) to cut windows.")
        print("(batch size is NOT a lever here -- see the sweep above)")
        print("Re-run §6 after editing §0 to see the new projection.")
        print("!" * 70)

[dispatch] warmup_bs32                  -> GPU 0   (2 queued, 1 running)
[dispatch] warmup_bs64                  -> GPU 1   (1 queued, 2 running)
[done]     warmup_bs32                  GPU 0    1.0 min | WAPE 32.721
[dispatch] warmup_bs128                 -> GPU 0   (0 queued, 2 running)
[done]     warmup_bs64                  GPU 1    1.2 min | WAPE 31.985
[done]     warmup_bs128                 GPU 0    1.8 min | WAPE 31.215

wall clock: 2.8 min  | 3 ok, 0 failed


,batch,ms_per_step,ms_per_sample,peak_gib,timed_steps,note
0,32,861.8,26.93,1.68,25,
1,64,1680.1,26.25,3.33,25,
2,128,3373.7,26.36,6.61,25,


params          : 963,376
train windows   : 9,984 (full dataset, fold 0)
configured batch: 32 at 26.93 ms/sample
fastest batch   : 64 at 26.25 ms/sample, peak 3.33 GiB of ~15 GiB
  -> set BATCH_SIZE = 64 in §0 and re-run from §3 for a 1.03x speedup (the projection below uses the CONFIGURED batch, not the fastest)
loss on original scale: True  (orig 2.7477 vs norm 0.5261, reported 2.7477)
causal/non-causal delta: 0.03379

--- projected REAL run (96 series, stride 24) ---
windows per fold: [9984, 11328, 12672] -> mean 11,328
354 steps/epoch @ batch 32 -> 5.08 min/epoch


,stage,runs,epochs,gpu_h,waves,wall_h
0,§11 final,5,10,4.24,3,2.54


TOTAL: 4.2 GPU-hours, ~2.5 h wall clock (quota 30 GPU-h/week; a session caps at 12 h)


---
## §7. Config probe

Four arms, each trained on `[0..2976)` and scored on the **inner** window `[2976..3312)`.
That window is disjoint from all three fold test blocks, so choosing a config here does not
contaminate the CV in §8.

| arm | what it tests |
|---|---|
| `base` | tutorial-faithful: fixed 336h encoder, causal attention, MAE |
| `noncausal` | + decoder attends to known future covariates |
| `crop` | + encoder-length randomisation (the exp-11 augmentation) |
| `quantile` | `crop` with `QuantileLoss` instead of `MAE` |

Four jobs across two GPUs = two waves. One 32k-row window resolves *structural* effects
(≳0.3 WAPE), **not** 0.05-level differences — read the arms as directional.

In [8]:
PROBE_ARMS = {
    "base":      dict(min_encoder=MAX_ENCODER, max_encoder=MAX_ENCODER, causal=True,  loss_name="mae"),
    "noncausal": dict(min_encoder=MAX_ENCODER, max_encoder=MAX_ENCODER, causal=False, loss_name="mae"),
    "crop":      dict(min_encoder=MIN_ENCODER, max_encoder=MAX_ENCODER, causal=False, loss_name="mae"),
    "quantile":  dict(min_encoder=MIN_ENCODER, max_encoder=MAX_ENCODER, causal=False, loss_name="quantile"),
}

assert len(PROBE_ARMS) == PROBE_ARMS_N, (
    f"PROBE_ARMS_N ({PROBE_ARMS_N}) is out of sync with PROBE_ARMS ({len(PROBE_ARMS)}) — "
    f"§6's budget projection prices the wrong number of probe runs")

if RUN_CONFIG_PROBE:
    probe_end = fold_train_end(0) - EVAL_HORIZON      # 2976
    jobs = [make_job(f"probe_{name}", train_end=probe_end, es_start=probe_end,
                     eval_start=probe_end, seed=SEED, max_epochs=PROBE_EPOCHS, **arm)
            for name, arm in PROBE_ARMS.items()]
    probe_res = run_jobs(jobs)

    probe = pd.DataFrame([
        dict(arm=name, wape=probe_res[f"probe_{name}"]["wape"],
             best_epoch=probe_res[f"probe_{name}"]["best_epoch"],
             val_loss=probe_res[f"probe_{name}"]["best_val_loss"],
             minutes=probe_res[f"probe_{name}"]["total_seconds"] / 60,
             windows=probe_res[f"probe_{name}"]["n_train_windows"],
             steps_ep=probe_res[f"probe_{name}"]["steps_per_epoch"])
        for name in PROBE_ARMS
    ]).sort_values("wape").reset_index(drop=True)
    probe.to_csv(PROBE_OUT, index=False)
    display(probe)

    best_arm = probe.iloc[0]["arm"]
    print(f"\nwinner: {best_arm}  (WAPE {probe.iloc[0]['wape']:.3f})")
    print(f"vs base: {probe.set_index('arm').loc['base', 'wape'] - probe.iloc[0]['wape']:+.3f} WAPE")
    print("\nIf the winner differs from the §0 defaults, update §0 (and re-run §3) before §8.")

---
## §8. Full cross-validation

3 folds × `SEEDS` = 9 jobs queued at once, drained two at a time. Per-fold predictions are
averaged across seeds (exp-11 protocol). Writes `oof_predictions_12a_tft.csv` in the
repository's standard OOF layout so it drops straight into the existing blend machinery.

In [9]:
oof, cv, epoch_log = None, None, []

if RUN_FULL_CV:
    jobs = []
    for k in range(N_FOLDS):
        te, es, oof_start = fold_bounds(k)
        for seed in SEEDS:
            # Keep exactly one checkpoint -- the last fold's first seed. §10 interprets that
            # model instead of training an identical one again, which saves a full run.
            jobs.append(make_job(f"cv_f{k}_s{seed}", train_end=te, es_start=es,
                                 eval_start=oof_start, seed=seed,
                                 save_ckpt=(k == N_FOLDS - 1 and seed == SEEDS[0])))
    cv_res = run_jobs(jobs)

    oof_parts, cv_rows = [], []
    for k in range(N_FOLDS):
        _, _, oof_start = fold_bounds(k)
        seed_preds, index = [], None
        for seed in SEEDS:
            tag = f"cv_f{k}_s{seed}"
            p, index = load_preds(tag)
            seed_preds.append(p)
            epoch_log.append(dict(fold=k, seed=seed, best_epoch=cv_res[tag]["best_epoch"],
                                  wape=cv_res[tag]["wape"],
                                  minutes=cv_res[tag]["total_seconds"] / 60))
        frame = preds_to_frame(np.mean(seed_preds, axis=0), index)
        frame["fold"] = k
        oof_parts.append(frame)
        w = wape(frame["target"], frame["pred_tft"])
        spread = float(np.std([e["wape"] for e in epoch_log if e["fold"] == k]))
        cv_rows.append(dict(fold=k, wape=w, seed_spread=spread))
        print(f"fold {k}: seed-averaged WAPE {w:.3f}  (seed spread {spread:.3f})")

    oof = pd.concat(oof_parts, ignore_index=True)[
        ["series_id", "timestamp", "target", "pred_tft", "lead_h", "fold"]]
    oof.to_csv(OOF_OUT, index=False)

    cv = pd.DataFrame(cv_rows)
    print()
    display(cv)
    display(pd.DataFrame(epoch_log))
    print(f"\nCV WAPE: {cv['wape'].mean():.3f} +/- {cv['wape'].std():.3f}"
          f"   (pooled {wape(oof['target'], oof['pred_tft']):.3f})")
    print(f"exp-11a GRU 14.203 +/- 0.599  |  exp-7c blend 14.47  |  our LB best 14.682")
    print(f"protocol: EARLY_STOP_MODE={EARLY_STOP_MODE}"
          + ("  (stricter than exp-11 — see README)" if EARLY_STOP_MODE == "inner" else ""))
    assert len(oof) == N_FOLDS * N_SERIES * EVAL_HORIZON, f"unexpected OOF row count: {len(oof)}"
    print(f"OOF rows: {len(oof):,} -> {OOF_OUT}")

---
## §9. Diagnostics

Per-fold, per-series, per-lead, and residual correlation against the existing models. The
correlation decides whether TFT is a *blend arm* even if it loses solo: the project gates
arms at `r < 0.85`, and everything tried so far has come back 0.96–0.99 (the "feature-bound
saturation" result). A materially lower `r` here is the first real crack in that finding.

In [10]:
if RUN_FULL_CV and oof is not None:
    # residual = pred - target, matching IMPROVEMENT-FRONTIERS.md: NEGATIVE = under-prediction.
    oof["residual"] = oof["pred_tft"] - oof["target"]

    print("— per fold —")
    display(oof.groupby("fold").apply(
        lambda g: pd.Series({"wape": wape(g.target, g.pred_tft), "mean_resid": g.residual.mean()}),
        include_groups=False))

    per_series = (oof.groupby("series_id")
                    .apply(lambda g: wape(g.target, g.pred_tft), include_groups=False)
                    .sort_values(ascending=False).rename("wape"))
    print("\n— worst 10 / best 5 series —")
    display(pd.concat([per_series.head(10), per_series.tail(5)]))
    print(f"spread {per_series.min():.2f} -> {per_series.max():.2f}   (exp-11a GRU: 7.01 -> 25.47)")

    lead_bucket = pd.cut(oof["lead_h"], [0, 24, 72, 168, 336],
                         labels=["1-24h", "25-72h", "73-168h", "169-336h"])
    print("\n— by forecast lead —")
    display(oof.groupby(lead_bucket, observed=True)
               .apply(lambda g: wape(g.target, g.pred_tft), include_groups=False).rename("wape"))

    hi = oof["target"] >= oof["target"].quantile(0.95)
    print(f"\ntop-5% target rows: WAPE {wape(oof.loc[hi,'target'], oof.loc[hi,'pred_tft']):.2f}, "
          f"mean residual {oof.loc[hi,'residual'].mean():+.2f}   (exp-7b blend: -5.60; "
          f"negative = under-shooting the spike)")

    print("\n— residual correlation vs prior models —")
    missing = []
    for name, fname in [("11a GRU",  "oof_predictions_11a_gru.csv"),
                        ("11b LSTM", "oof_predictions_11b_lstm.csv"),
                        ("7b blend", "oof_predictions_7b_blend.csv")]:
        path = find_prior(fname)
        if path is None:
            missing.append(name)
            continue
        other = pd.read_csv(path, parse_dates=["timestamp"])
        pred_col = pick_pred_col(other, path)
        print(f"  [{name}] {path}  (column {pred_col})")
        other["series_id"] = other["series_id"].astype(str)
        m = oof.merge(other[["series_id", "timestamp", pred_col]], on=["series_id", "timestamp"])
        r = float(np.corrcoef(m["residual"], m[pred_col] - m["target"])[0, 1])
        verdict = "STRONG diversity" if r < 0.85 else ("some diversity" if r < 0.95 else "saturated")
        print(f"  r vs {name:<9}: {r:.4f}  [{verdict}]  ({len(m):,} rows matched)")

    if missing:
        print("\n" + "!" * 70)
        print(f"NOT COMPUTED for: {', '.join(missing)} — prior OOF files were not found.")
        print("The residual correlation is the pre-registered gate for shipping TFT as a")
        print("BLEND ARM (r < 0.85 strong, > 0.95 saturated). Without it §12 also has only")
        print("one arm and cannot blend. Attach experiment-7/ and experiment-11/ CSVs as a")
        print("Kaggle dataset and re-run this cell — it needs no GPU.")
        print("!" * 70)

---
## §10. Variable selection — what the model actually uses

exp-8 rejected TiDE partly on a diagnostic: its importances showed `target_history`
dominating at 0.42–0.49, i.e. it leaned autoregressive instead of exploiting the future
covariates that are the whole point of this dataset. TFT reports this natively and separately
for the encoder and decoder pathways.

The decoder half is the interesting one — those are the *known future* inputs, the part no
tree-GBM on lag features can see. This runs in-kernel on one GPU (single short job).

In [11]:
if RUN_FULL_CV:
    import matplotlib.pyplot as plt
    from lightning.pytorch import seed_everything
    from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet, GroupNormalizer
    from pytorch_forecasting.metrics import MAE, QuantileLoss

    sys.path.insert(0, str(RUN_DIR))
    import importlib, tft_worker
    importlib.reload(tft_worker)

    te, es, oof_start = fold_bounds(N_FOLDS - 1)

    def find_ckpt(tag):
        """Exact `{tag}.ckpt`, else the newest `{tag}*.ckpt` (Lightning appends -v1 on repeats)."""
        exact = RUN_DIR / "ckpt" / f"{tag}.ckpt"
        if exact.exists():
            return exact
        cands = sorted((RUN_DIR / "ckpt").glob(f"{tag}*.ckpt"), key=lambda q: q.stat().st_mtime)
        return cands[-1] if cands else None

    # §8 already trained this exact model (last fold, first seed) and kept its checkpoint.
    # Re-training it here would burn a full run on one GPU while the other idles.
    cv_tag = f"cv_f{N_FOLDS - 1}_s{SEEDS[0]}"
    ckpt = find_ckpt(cv_tag)
    if ckpt is None:
        print(f"no checkpoint for {cv_tag} (was §8 run?) — training one interpretation model")
        _ = run_jobs([make_job("interp", train_end=te, es_start=es, eval_start=oof_start,
                               seed=SEEDS[0], save_ckpt=True)], n_gpus=1)
        ckpt = find_ckpt("interp")
    else:
        print(f"interpreting the §8 checkpoint {ckpt.name} — no extra training run")

    model = TemporalFusionTransformer.load_from_checkpoint(str(ckpt)).to("cuda:0")

    train_ds = tft_worker.build_train_ds(panel, CFG, te)
    eval_ds  = tft_worker.build_eval_ds(train_ds, panel, CFG, oof_start)
    raw = model.predict(tft_worker.to_dl(eval_ds, CFG, train=False), mode="raw", return_x=True,
                        trainer_kwargs=dict(accelerator="gpu", devices=1, logger=False,
                                            enable_progress_bar=False))
    interp = model.interpret_output(raw.output, reduction="sum")

    rows = []
    for scope, names in [("static", model.static_variables),
                         ("encoder", model.encoder_variables),
                         ("decoder", model.decoder_variables)]:
        v = interp[f"{scope}_variables"].detach().cpu().float().numpy()
        v = v / v.sum()
        rows += [dict(scope=scope, variable=n, importance=float(x)) for n, x in zip(names, v)]
    imp = pd.DataFrame(rows).sort_values(["scope", "importance"], ascending=[True, False])
    imp.to_csv(IMPORTANCE_OUT, index=False)

    for scope in ("static", "encoder", "decoder"):
        print(f"\n— {scope} variable importance (top 8) —")
        display(imp[imp.scope == scope].head(8).set_index("variable")["importance"])

    tgt = imp[(imp.scope == "encoder") & (imp.variable == "target")]
    if len(tgt):
        print(f"\ntarget_history share of encoder importance: {tgt.importance.iloc[0]:.3f}"
              f"   (TiDE, rejected in exp-8: 0.42-0.49)")

    model.plot_interpretation(interp)
    attn = interp["attention"].detach().cpu().float().numpy()
    plt.figure(figsize=(10, 3))
    plt.plot(attn)
    plt.title("Attention by relative time index (negative = encoder history)")
    plt.xlabel("relative position"); plt.ylabel("attention")
    plt.tight_layout(); plt.show()

---
## §11. Final fit & submission

**This is where exp-11 lost 3.1 WAPE.** Bug 1 in `experiment-11/README.md`: the final model
trained on windows that stopped ~1200h short of the forecast origin, so at inference the
encoder read a region it had never seen. LB went 17.912 → 14.773 on the fix.

The guarantee here is that training windows must reach `T_TRAIN` — the last training decoder
window ends exactly at 4320. That puts the last training *encoder* at [3648..3984) while
inference reads [3984..4320): a 336h gap, which is the **structural minimum** for any model
trained with a full-horizon decoder (the last encoder must sit one whole horizon back so its
decoder has targets). The failure mode is a gap *larger* than that.

To reach the minimum the final fit uses **fixed epochs** (the CV median best epoch) rather
than early stopping — early stopping would need the last 336h held out, pushing the gap to
672h. Both assertions below are guards.

In [12]:
if RUN_FINAL_FIT:
    if FINAL_EPOCHS_OVERRIDE:
        final_epochs = int(FINAL_EPOCHS_OVERRIDE)
        print(f"final fit: {final_epochs} epochs (FINAL_EPOCHS_OVERRIDE — §8 not required)")
    elif epoch_log:
        final_epochs = int(np.median([e["best_epoch"] for e in epoch_log]))
    else:
        raise RuntimeError(
            "§11 needs an epoch count: either run §8 (RUN_FULL_CV=True) or set "
            "FINAL_EPOCHS_OVERRIDE in §0 to the CV median best_epoch (run 4: 10).")
    if not FINAL_EPOCHS_OVERRIDE:
        print(f"final fit: {final_epochs} epochs (CV median best epoch), no early stopping")

    jobs = [make_job(f"final_s{seed}", train_end=T_TRAIN, es_start=None, eval_start=T_TRAIN,
                     seed=seed, fixed_epochs=final_epochs, save_ckpt=True)
            for seed in SEEDS]
    final_res = run_jobs(jobs)

    # exp-11 Bug 1 guard: training windows must reach the forecast origin
    fr = final_res[f"final_s{SEEDS[0]}"]
    gap = T_TRAIN - fr["last_train_encoder_end"]
    print(f"\nlast training decoder target : {fr['last_train_target']} (must be {T_TRAIN - 1})")
    print(f"last training encoder end    : {fr['last_train_encoder_end']}")
    print(f"inference encoder end        : {T_TRAIN}")
    print(f"encoder distribution gap     : {gap}h "
          f"(structural minimum {EVAL_HORIZON}h; exp-11's bug made it ~1200h)")
    assert fr["last_train_target"] == T_TRAIN - 1, \
        "training windows stop short of the forecast origin — this is exp-11 Bug 1"
    assert gap <= EVAL_HORIZON, "encoder gap exceeds the horizon"

    seed_preds, index = [], None
    for seed in SEEDS:
        p, index = load_preds(f"final_s{seed}")
        seed_preds.append(p)
    assert (index["time_idx"] == T_TRAIN).all(), "submission window misaligned"

    sub = preds_to_frame(np.mean(seed_preds, axis=0), index).rename(columns={"pred_tft": "prediction"})
    sub = sub[["series_id", "timestamp", "prediction"]].sort_values(["series_id", "timestamp"])

    expected = val_index[["series_id", "timestamp"]].copy()
    expected["series_id"] = expected["series_id"].astype(str)
    check = expected.merge(sub, on=["series_id", "timestamp"], how="left")
    assert check["prediction"].notna().all(), f"{check.prediction.isna().sum()} rows missing vs forecast index"
    assert len(sub) == N_SERIES * EVAL_HORIZON, f"expected {N_SERIES*EVAL_HORIZON} rows, got {len(sub)}"
    assert np.isfinite(sub["prediction"]).all(), "non-finite predictions"

    sub.to_csv(VAL_OUT, index=False)
    print(f"\nwrote {VAL_OUT}  ({len(sub):,} rows)")
    display(sub["prediction"].describe())
    print("\ntraining target distribution for comparison:")
    display(panel.loc[~panel.is_future, "target"].describe())

final fit: 10 epochs (FINAL_EPOCHS_OVERRIDE — §8 not required)
[dispatch] final_s42                    -> GPU 0   (4 queued, 1 running)
[dispatch] final_s7                     -> GPU 1   (3 queued, 2 running)
[done]     final_s42                    GPU 0   69.7 min
[done]     final_s7                     GPU 1   69.7 min
[dispatch] final_s13                    -> GPU 0   (2 queued, 1 running)
[dispatch] final_s99                    -> GPU 1   (1 queued, 2 running)
[done]     final_s13                    GPU 0   69.7 min
[done]     final_s99                    GPU 1   69.7 min
[dispatch] final_s1234                  -> GPU 0   (0 queued, 1 running)
[done]     final_s1234                  GPU 0   69.5 min

wall clock: 208.8 min  | 5 ok, 0 failed

last training decoder target : 4319 (must be 4319)
last training encoder end    : 3984
inference encoder end        : 4320
encoder distribution gap     : 336h (structural minimum 336h; exp-11's bug made it ~1200h)

wrote /kaggle/working/validati

count    32256.000000
mean        10.397417
std          5.266882
min          0.942969
25%          6.406250
50%          9.587500
75%         13.600000
max         36.599998
Name: prediction, dtype: float64


training target distribution for comparison:


count    414720.000000
mean          9.912838
std           5.547878
min           0.164003
25%           5.801159
50%           9.041956
75%          13.025928
max          53.000164
Name: target, dtype: float64

---
## §12. Blend hook

The existing arms (`11a` GRU, `7c` LGBM+MLP) are already OOF-aligned, so a simplex-weighted
blend is a short addition once §8 has run. Weights are fit on OOF and applied to the
validation components — never fit on the leaderboard.

Attach the prior experiments' OOF + validation CSVs as a Kaggle dataset and point
`BLEND_SOURCES` at them.

In [13]:
from scipy.optimize import minimize

if not RUN_BLEND:
    print("§12 skipped (RUN_BLEND=False) — blending is stage 0, on matched inputs")

if RUN_BLEND:

    # Resolved by filename, so it does not matter what the Kaggle dataset is called or how its
    # folders are nested.
    BLEND_SOURCES = {
        "tft": (OOF_OUT, VAL_OUT),
        "gru": (find_prior("oof_predictions_11a_gru.csv"),
                find_prior("validation_predictions_11a_gru.csv")),
        "7c":  (find_prior("oof_predictions_7b_blend.csv"),
                find_prior("validation_predictions_7c.csv")),
    }

    for k, (o, v) in BLEND_SOURCES.items():
        print(f"  {k:5s} oof={o}  val={v}")
    available = {k: v for k, v in BLEND_SOURCES.items()
                 if v[0] is not None and v[1] is not None and Path(v[0]).exists() and Path(v[1]).exists()}
    print(f"blend arms available: {list(available)}")
    if len(available) < 2:
        print("\n" + "!" * 70)
        print("Only one arm — §12 cannot blend. Attach the prior-experiment CSVs as a Kaggle")
        print("dataset (any name; they are found by filename). See experiment-12/kaggle_priors/.")
        print("!" * 70)

    if len(available) >= 2:
        oof_m, val_m = None, None
        for name, (op, vp) in available.items():
            o = pd.read_csv(op, parse_dates=["timestamp"])
            pc = pick_pred_col(o, op)
            o = o[["series_id", "timestamp", "target", pc]].rename(columns={pc: name})
            o["series_id"] = o["series_id"].astype(str)
            oof_m = o if oof_m is None else oof_m.merge(o.drop(columns="target"),
                                                        on=["series_id", "timestamp"])
            v = pd.read_csv(vp, parse_dates=["timestamp"])
            vc = [c for c in v.columns if c in ("prediction", "pred")][0]
            v = v[["series_id", "timestamp", vc]].rename(columns={vc: name})
            v["series_id"] = v["series_id"].astype(str)
            val_m = v if val_m is None else val_m.merge(v, on=["series_id", "timestamp"])

        arms = list(available)
        X, y = oof_m[arms].values, oof_m["target"].values
        print(f"{len(oof_m):,} OOF rows matched across arms")
        for a in arms:
            print(f"  {a:5s} solo OOF WAPE {wape(y, oof_m[a]):.3f}")

        def obj(w):
            w = np.clip(w, 0, None)
            return wape(y, X @ (w / (w.sum() + 1e-12)))

        res = minimize(obj, np.full(len(arms), 1 / len(arms)), method="Nelder-Mead",
                       options=dict(maxiter=4000, xatol=1e-6, fatol=1e-8))
        w = np.clip(res.x, 0, None); w = w / w.sum()
        print("\nweights:", {a: round(float(x), 4) for a, x in zip(arms, w)})
        print(f"blend OOF WAPE: {obj(res.x):.3f}")

        val_m["prediction"] = val_m[arms].values @ w
        out = OUTPUT_DIR / f"validation_predictions_12b_blend{'_preflight' if PREFLIGHT else ''}.csv"
        val_m[["series_id", "timestamp", "prediction"]].to_csv(out, index=False)
        print(f"wrote {out}  ({len(val_m):,} rows)")

§12 skipped (RUN_BLEND=False) — blending is stage 0, on matched inputs


---
## §13. Hyperparameter tuning (optional) — both GPUs

Off by default, and **not where the leverage is**: the §7 arms test structural choices worth
~0.3–1.5 WAPE each, while `hidden_size` 64 vs 96 is worth a fraction of that. Tune only after
§7 and §8 have converged and quota remains.

This drives Optuna in **ask-and-tell** mode: the sampler proposes `N_GPUS` configs at a time
from the notebook (cheap, CPU-side), the §5 dispatcher trains them **concurrently, one per
GPU**, and the resulting WAPEs are told back to the sampler. So you get TPE's sample
efficiency *and* both cards busy.

It deliberately does **not** use `pytorch_forecasting`'s `optimize_hyperparameters`, which
was the earlier version of this cell:

| | `optimize_hyperparameters` | this cell |
|---|---|---|
| GPUs | one — the study loop is sequential | **both**, via the dispatcher |
| objective | `val_loss` | **fold WAPE**, computed exactly as §7/§8 do |
| training path | its own internal Trainer | the same worker used everywhere else |
| extra installs | optuna + optuna-integration + statsmodels + tensorboardX | optuna only (`"random"` needs none) |
| silent-default traps | `loss`, `causal_attention`, `mask_bias` must all be hand-forwarded or it tunes the wrong model | none — the config goes through `make_job` |

Tuning stays on the inner window `[2976..3312)`, disjoint from every fold test block, so it
cannot contaminate the CV. Set `TUNE_SAMPLER = "random"` to drop the optuna dependency
entirely.

A caution on reading the table: one 32k-row window resolves ≳0.3 WAPE. Differences below that
between the top few configs are noise — take the *region* of the space that does well, not the
single best row, and confirm anything on the full CV in §8 before adopting it.

In [14]:
TUNE_TRIALS  = TUNE_TRIALS_OVERRIDE or 8
TUNE_EPOCHS  = MAX_EPOCHS if PREFLIGHT else 25   # ranking, not final fitting
TUNE_SAMPLER = "tpe"                             # "tpe" (needs optuna) | "random" (no deps)

SPACE = dict(
    hidden_size=[64, 96, 128],
    hidden_continuous_size=[16, 32],
    attention_heads=[2, 4],
    lstm_layers=[1, 2],
    dropout=[0.10, 0.15, 0.20, 0.25],
    learning_rate=[3e-4, 1e-3, 3e-3],
)

if RUN_TUNING:
    probe_end = fold_train_end(0) - EVAL_HORIZON
    use_tpe = TUNE_SAMPLER == "tpe"

    if use_tpe:
        try:
            import optuna
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
                            "optuna"], check=True)
            import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(direction="minimize",
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
    else:
        rng = np.random.default_rng(SEED)

    print(f"{TUNE_TRIALS} trials, sampler={TUNE_SAMPLER}, {TUNE_EPOCHS} epochs each, "
          f"{N_GPUS} at a time\n")

    rows, done = [], 0
    while done < TUNE_TRIALS:
        k = min(N_GPUS, TUNE_TRIALS - done)          # one proposal per free GPU
        trials, picks, jobs = [], [], []
        for j in range(k):
            if use_tpe:
                t = study.ask()
                pick = {n: t.suggest_categorical(n, v) for n, v in SPACE.items()}
                trials.append(t)
            else:
                pick = {n: v[int(rng.integers(len(v)))] for n, v in SPACE.items()}
            picks.append(pick)
            jobs.append(make_job(f"tune_{done + j:02d}", train_end=probe_end,
                                 es_start=probe_end, eval_start=probe_end, seed=SEED,
                                 max_epochs=TUNE_EPOCHS, patience=5, **pick))

        res = run_jobs(jobs)                          # <- both GPUs busy

        for j, (job, pick) in enumerate(zip(jobs, picks)):
            w = res[job["tag"]]["wape"]
            if use_tpe:
                study.tell(trials[j], w)
            rows.append(dict(tag=job["tag"], wape=w,
                             best_epoch=res[job["tag"]]["best_epoch"],
                             minutes=round(res[job["tag"]]["total_seconds"] / 60, 1), **pick))
        done += k

    tune = pd.DataFrame(rows).sort_values("wape").reset_index(drop=True)
    tune.to_csv(OUTPUT_DIR / f"tuning_{EXP_ID}.csv", index=False)
    display(tune)

    best = tune.iloc[0]
    print(f"\nbest: {best['wape']:.3f} WAPE")
    for n in SPACE:
        print(f"  {n:24s} {best[n]}")
    print(f"\nspread across trials: {tune['wape'].min():.3f} -> {tune['wape'].max():.3f} "
          f"(range {tune['wape'].max() - tune['wape'].min():.3f})")
    print("Differences < ~0.3 WAPE on this single window are noise — prefer the region that")
    print("does well over the single best row, and confirm any change on the full CV in §8.")

---
## §14. Package results — one zip, one report

Two artifacts land in `/kaggle/working`, and nothing else does (intermediates live in
scratch, see §0):

- **`run_report_<EXP_ID>.md`** — a compact, self-contained digest of the whole run: config,
  premise checks, throughput, probe table, per-fold and per-job CV, per-series and per-lead
  WAPE, spike-tail stats, residual correlations, variable importance, tuning, final-fit
  guards, submission stats, blend weights, timings. Typically 10–30 KB — small enough to
  open in the browser or paste straight into a chat for analysis.
- **`experiment12_<EXP_ID>_<UTC>.zip`** — one download containing the report, every result
  CSV, and the full run tree (job specs, worker logs, per-job result JSONs, raw `.npz`
  predictions, and the worker script itself).

Checkpoints are excluded by default (~10 MB each and useless for analysis); flip
`INCLUDE_CKPT` if you want them. `PRUNE_RUN_DIR` deletes the scratch tree after zipping —
off by default so a failed run keeps its logs on disk for inspection.

In [15]:
import platform, shutil, zipfile
from datetime import datetime, timezone

PACKAGE_RESULTS = True
INCLUDE_NPZ     = True      # raw per-job predictions, ~130 KB each
INCLUDE_CKPT    = False     # model checkpoints, ~10 MB each
PRUNE_RUN_DIR   = False     # delete the scratch tree after zipping

REPORT_PATH = OUTPUT_DIR / f"run_report_{EXP_ID}.md"
L = []                                            # report lines
def w(s=""): L.append(s)
def block(df, n=None):
    w("```")
    w((df.head(n) if n else df).to_string())
    w("```")

def section(title, fn):
    w(f"\n## {title}\n")
    try:
        fn()
    except Exception as e:
        w(f"*(section failed: {type(e).__name__}: {e})*")

def _results():
    return [json.loads(q.read_text()) for q in sorted((RUN_DIR / "results").glob("*.json"))]

if PACKAGE_RESULTS:
    recs = _results()
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

    w(f"# Run report — {EXP_ID}")
    w(f"\n`EXP_ID={EXP_ID}` · generated {stamp} · **PREFLIGHT={PREFLIGHT}**")
    if PREFLIGHT:
        w(f"\n> **PREFLIGHT run — every WAPE below is meaningless by design** "
          f"({MAX_EPOCHS} epoch(s), stride {STRIDE}, {len(SEEDS)} seed(s), "
          f"{N_SERIES} of {N_SERIES_FULL} series). This report exists to show the pipeline "
          f"ran, not to be read for accuracy.")

    def _env():
        w(f"- python `{platform.python_version()}` · torch `{torch.__version__}` · "
          f"lightning `{pl.__version__}` · pytorch-forecasting `{pf.__version__}`")
        w(f"- GPUs visible: {N_GPUS}" + (f" — {torch.cuda.get_device_name(0)}" if N_GPUS else ""))
        if recs:
            w(f"- jobs run: {len(recs)} · total GPU time "
              f"{sum(r['total_seconds'] for r in recs)/3600:.2f} h")
            span = max(r["finished_at"] for r in recs) - min(r["started_at"] for r in recs)
            w(f"- wall clock across all jobs: {span/3600:.2f} h "
              f"(speedup {sum(r['total_seconds'] for r in recs)/max(span,1e-9):.2f}x)")
    section("Environment", _env)

    def _cfg():
        keys = ["min_encoder", "max_encoder", "randomize_length", "stride", "hidden_size",
                "hidden_continuous_size", "attention_heads", "lstm_layers", "dropout",
                "causal", "share_vsn", "loss_name", "learning_rate", "weight_decay",
                "optimizer", "grad_clip", "batch_size", "accum", "max_epochs", "patience"]
        for k in keys:
            w(f"- `{k}` = `{CFG[k]}`")
        w(f"- `EARLY_STOP_MODE` = `{EARLY_STOP_MODE}` · `SEEDS` = `{SEEDS}` · "
          f"`N_FOLDS` = {N_FOLDS} · `EVAL_HORIZON` = {EVAL_HORIZON}")
        if recs:
            w(f"- model parameters: {recs[0]['n_params']:,}")
    section("Configuration", _cfg)

    def _premises():
        # only the kind="throughput" warmup runs the premise checks; sorted() puts
        # warmup_bs128 first, which does not, so select by kind rather than by order
        warm = [r for r in recs if r.get("kind") == "throughput"]
        if not warm:
            w("*(throughput stage not run)*"); return
        r = warm[0]
        w(f"- loss on ORIGINAL target scale: **{r.get('loss_on_original_scale')}** "
          f"(original {r.get('mae_original', float('nan')):.4f} vs "
          f"normalised {r.get('mae_normalized', float('nan')):.4f}, "
          f"reported val_loss {r.get('reported_val_loss', float('nan')):.4f})")
        w(f"- causal vs non-causal mean |Δprediction|: **{r.get('causal_delta', float('nan')):.5f}**")
        w(f"- {r['steps_per_epoch']} steps/epoch · "
          f"{r['sec_per_epoch']:.1f} s per measured epoch · {r['n_train_windows']:,} train windows")
    section("Premise checks & throughput", _premises)

    def _probe():
        if not Path(PROBE_OUT).exists():
            w("*(config probe not run)*"); return
        block(pd.read_csv(PROBE_OUT))
    section("§7 config probe", _probe)

    def _cv():
        if not Path(OOF_OUT).exists():
            w("*(cross-validation not run)*"); return
        o = pd.read_csv(OOF_OUT, parse_dates=["timestamp"])
        per_fold = o.groupby("fold").apply(
            lambda g: pd.Series({"wape": wape(g.target, g.pred_tft),
                                 "mean_resid": g.pred_tft.sub(g.target).mean(),
                                 "rows": len(g)}), include_groups=False)
        block(per_fold)
        w(f"\n**CV WAPE {per_fold['wape'].mean():.3f} ± {per_fold['wape'].std():.3f}** "
          f"(pooled {wape(o.target, o.pred_tft):.3f}, {len(o):,} rows)")
        w(f"\nReference: exp-11a GRU 14.203 ± 0.599 · exp-7c blend 14.47 · our LB best 14.682")
        cvj = [r for r in recs if r["tag"].startswith("cv_")]
        if cvj:
            w("\nPer job:")
            block(pd.DataFrame([{ "tag": r["tag"], "seed": r["seed"], "wape": r.get("wape"),
                                  "best_epoch": r["best_epoch"], "epochs_run": r["epochs_run"],
                                  "minutes": round(r["total_seconds"]/60, 1)} for r in cvj]))
    section("§8 cross-validation", _cv)

    def _diag():
        if not Path(OOF_OUT).exists():
            w("*(cross-validation not run)*"); return
        o = pd.read_csv(OOF_OUT, parse_dates=["timestamp"])
        o["residual"] = o.pred_tft - o.target      # negative = under-prediction
        ps = (o.groupby("series_id").apply(lambda g: wape(g.target, g.pred_tft),
                                           include_groups=False)
                .sort_values(ascending=False).rename("wape"))
        w("Worst 15 / best 5 series (exp-11a GRU spread: 7.01 → 25.47):")
        block(pd.concat([ps.head(15), ps.tail(5)]).to_frame())
        lb = pd.cut(o.lead_h, [0, 24, 72, 168, 336], labels=["1-24h","25-72h","73-168h","169-336h"])
        w("\nBy forecast lead:")
        block(o.groupby(lb, observed=True).apply(lambda g: wape(g.target, g.pred_tft),
                                                 include_groups=False).rename("wape").to_frame())
        hi = o.target >= o.target.quantile(0.95)
        w(f"\nTop-5% target rows: WAPE {wape(o.loc[hi,'target'], o.loc[hi,'pred_tft']):.2f}, "
          f"mean residual {o.loc[hi,'residual'].mean():+.2f} (exp-7b blend reference: −5.60)")
        w(f"\nOverall mean residual {o.residual.mean():+.3f}")
        if "BLEND_SOURCES" in globals():
            w("\nResidual correlation vs prior models:")
            for nm, (op, _) in BLEND_SOURCES.items():
                if nm == "tft" or not Path(op).exists():
                    continue
                other = pd.read_csv(op, parse_dates=["timestamp"])
                pc = pick_pred_col(other, op)
                other["series_id"] = other["series_id"].astype(str)
                mg = o.merge(other[["series_id","timestamp",pc]], on=["series_id","timestamp"])
                r = float(np.corrcoef(mg.residual, mg[pc] - mg.target)[0, 1])
                verdict = "STRONG" if r < .85 else ("some" if r < .95 else "saturated")
                w(f"- vs `{nm}`: r = **{r:.4f}** [{verdict}] ({len(mg):,} rows)")
    section("§9 diagnostics", _diag)

    def _imp():
        if not Path(IMPORTANCE_OUT).exists():
            w("*(interpretation not run)*"); return
        imp = pd.read_csv(IMPORTANCE_OUT)
        for sc in ("static", "encoder", "decoder"):
            w(f"\n**{sc}**")
            block(imp[imp.scope == sc].nlargest(10, "importance")[["variable","importance"]])
        t = imp[(imp.scope == "encoder") & (imp.variable == "target")]
        if len(t):
            w(f"\n`target_history` share of encoder importance: **{t.importance.iloc[0]:.3f}** "
              f"(TiDE, rejected in exp-8: 0.42–0.49)")
    section("§10 variable selection", _imp)

    def _tune():
        tp = OUTPUT_DIR / f"tuning_{EXP_ID}.csv"
        if not tp.exists():
            w("*(tuning not run)*"); return
        block(pd.read_csv(tp))

    def _final():
        fin = [r for r in recs if r["tag"].startswith("final_")]
        if not fin:
            w("*(final fit not run)*"); return
        r = fin[0]
        w(f"- last training decoder target: **{r['last_train_target']}** (must be {T_TRAIN-1})")
        w(f"- last training encoder end: {r['last_train_encoder_end']} → "
          f"inference encoder gap **{T_TRAIN - r['last_train_encoder_end']}h** "
          f"(structural minimum {EVAL_HORIZON}h)")
        w(f"- epochs (fixed, from CV median): {r['epochs_run']} · seeds averaged: {len(fin)}")
        if Path(VAL_OUT).exists():
            s = pd.read_csv(VAL_OUT, parse_dates=["timestamp"])
            w(f"\nSubmission `{Path(VAL_OUT).name}` — {len(s):,} rows, "
              f"{s.timestamp.min()} → {s.timestamp.max()}")
            block(s.prediction.describe().to_frame())
        bl = OUTPUT_DIR / "validation_predictions_12b_blend.csv"
        if bl.exists():
            w(f"\nBlend submission written: `{bl.name}` ({len(pd.read_csv(bl)):,} rows)")
    section("§11 final fit & submission", _final)
    section("§13 tuning", _tune)

    def _timing():
        if not recs:
            w("*(no jobs)*"); return
        block(pd.DataFrame([{ "tag": r["tag"], "minutes": round(r["total_seconds"]/60, 1),
                              "epochs": r["epochs_run"], "sec/epoch": round(r["sec_per_epoch"], 1),
                              "wape": r.get("wape")} for r in recs]).sort_values("minutes",
                                                                                ascending=False))
    section("Job timings", _timing)

    REPORT_PATH.write_text("\n".join(L))

    # ── one zip ───────────────────────────────────────────────────────────────
    stamp_f = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    zip_path = OUTPUT_DIR / f"results_{EXP_ID}_{stamp_f}.zip"
    n_files = 0
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as z:
        for q in sorted(OUTPUT_DIR.glob("*.csv")):
            z.write(q, f"outputs/{q.name}"); n_files += 1
        z.write(REPORT_PATH, f"outputs/{REPORT_PATH.name}"); n_files += 1
        for sub in ("jobs", "logs", "results"):
            for q in sorted((RUN_DIR / sub).glob("*")):
                if q.suffix == ".npz" and not INCLUDE_NPZ:
                    continue
                z.write(q, f"run/{sub}/{q.name}"); n_files += 1
        if INCLUDE_CKPT:
            for q in sorted(MODEL_DIR.glob("*.ckpt")):
                z.write(q, f"models/{q.name}"); n_files += 1
        if WORKER.exists():
            z.write(WORKER, "run/tft_worker.py"); n_files += 1

    print(f"report : {REPORT_PATH}  ({REPORT_PATH.stat().st_size/1024:.0f} KB)")
    print(f"bundle : {zip_path}  ({zip_path.stat().st_size/2**20:.1f} MB, {n_files} files)")
    print("\nDownload the two files above from /kaggle/working — the zip has everything else.")
    print("For analysis, the .md report alone is usually enough to paste into a chat.")

    models = sorted(MODEL_DIR.glob("*.ckpt")) if MODEL_DIR.exists() else []
    if models:
        tot = sum(q.stat().st_size for q in models) / 2**20
        print(f"\nmodels : {len(models)} checkpoint(s), {tot:.0f} MB total, in {MODEL_DIR}")
        for q in models:
            print(f"  {q.stat().st_size/2**20:6.1f} MB  {q.name}")
        print("  (excluded from the zip unless INCLUDE_CKPT — download separately if wanted)")
    else:
        print("\n!! No model checkpoints were saved — the final fit is unreproducible.")

    if PRUNE_RUN_DIR:
        shutil.rmtree(RUN_DIR, ignore_errors=True)
        print(f"\npruned scratch tree {RUN_DIR}")

    print(f"\n/kaggle/working now contains:")
    for q in sorted(OUTPUT_DIR.iterdir()):
        if q.is_file():
            print(f"  {q.stat().st_size/1024:9.0f} KB  {q.name}")
        else:
            print(f"  {'<dir>':>9}      {q.name}/")

report : /kaggle/working/run_report_13a_tft_multiseed.md  (3 KB)
bundle : /kaggle/working/results_13a_tft_multiseed_20260817_103616.zip  (0.6 MB, 35 files)

Download the two files above from /kaggle/working — the zip has everything else.
For analysis, the .md report alone is usually enough to paste into a chat.

models : 5 checkpoint(s), 60 MB total, in /kaggle/working/models
    12.0 MB  final_s1234.ckpt
    12.0 MB  final_s13.ckpt
    12.0 MB  final_s42.ckpt
    12.0 MB  final_s7.ckpt
    12.0 MB  final_s99.ckpt
  (excluded from the zip unless INCLUDE_CKPT — download separately if wanted)

/kaggle/working now contains:
      <dir>      .virtual_documents/
      <dir>      models/
        612 KB  results_13a_tft_multiseed_20260817_103616.zip
          3 KB  run_report_13a_tft_multiseed.md
       1425 KB  validation_predictions_13a_tft_multiseed.csv
